# Agent Memory Server - 25 Minute Interactive Demo

This notebook focuses on: setup, code-driven memory, working memory, background extraction, search, and one episodic-memory example.


## Quick Start

Before running this notebook, make sure you have:

1. **Redis running**
2. **AMS running**
3. **`OPENAI_API_KEY` set**
4. **Python dependencies installed** (`agent-memory-client`, `openai`, `python-dotenv`)

This is designed for roughly **20-25 minutes live**.


You can get a local Redis instance running, as well as a local Agent Memory Server by running the included docker `compose.yaml` file.
```
docker compose up -d
```

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import required libraries
import os
from datetime import datetime, timezone
from dotenv import load_dotenv


# Load environment variables from .env file
load_dotenv()

# Configuration
BASE_URL = "http://localhost:8000"
NAMESPACE = "travel_agent"

print(f"Base URL: {BASE_URL}")
print(f"Namespace: {NAMESPACE}")
print(f"OpenAI API Key: {'loaded' if os.environ.get('OPENAI_API_KEY') else '✗ not found'}")

Base URL: http://localhost:8000
Namespace: travel_agent
OpenAI API Key: loaded


In [3]:
# Initialize the Python SDK client
from agent_memory_client import MemoryAPIClient, MemoryClientConfig

config = MemoryClientConfig(
    base_url=BASE_URL,
    timeout=30.0,
    default_namespace=NAMESPACE,  # "travel_agent"
)

client = MemoryAPIClient(config)
print("Memory client initialized!")
print("health check (current time):",await client.health_check())
print(f"Default namespace: {config.default_namespace}")

Memory client initialized!
health check (current time): now=1775029456530.0
Default namespace: travel_agent


## Demo Agenda

This notebook covers:

1. Code-driven memory with `memory_prompt()`
2. Writing and inspecting working memory
3. Background extraction
4. Long-term search
5. episodic-memory


# Pattern 1: Code-Driven Memory

Manually store and retrieve memories about a user session.\
Use this when your application code should decide exactly when to read or write memory.

### Step 1: Initialize memory client

In [4]:
# Pattern 1 Code driven

from agent_memory_client.models import (
    ClientMemoryRecord,
    MemoryMessage,
    MemoryTypeEnum,
    WorkingMemory,
)


# Use "nitin" as our demo user
SESSION_ID = "nitin-travel-session"
USER_ID = "nitin"

# Step 1: Create/get a working memory session for Nitin
created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID, namespace=NAMESPACE, user_id=USER_ID
)

print(f"Session {'created' if created else 'already existed'}: {SESSION_ID}")
print(f"User: {USER_ID}")
print(f"Namespace: {NAMESPACE}")

Session created: nitin-travel-session
User: nitin
Namespace: travel_agent


### Step 2: Seed data into longterm memory

In [5]:
# Nitin's travel preferences (from the travel agent demo)
memories_to_store = [
    ClientMemoryRecord(
        text="Comfort travel - middle tier (not luxury, not budget)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Vegetarian diet - needs vegetarian restaurant options",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Extra leg room on flights (premium economy or exit row and budget allows, anything goes)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Prefers hotels with good amenities",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Enjoys technology, sports, outdoors, and innovation hubs",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
]

print(f"Storing {len(memories_to_store)} preferences for user '{USER_ID}'...")
result = await client.create_long_term_memory(memories_to_store)
print(f"Result: {result.status}")

# Show what we just stored
print("\nStored memories:")
for i, mem in enumerate(memories_to_store, 1):
    print(f"  {i}. {mem.text}")

Storing 5 preferences for user 'nitin'...
Result: ok

Stored memories:
  1. Comfort travel - middle tier (not luxury, not budget)
  2. Vegetarian diet - needs vegetarian restaurant options
  3. Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
  4. Prefers hotels with good amenities
  5. Enjoys technology, sports, outdoors, and innovation hubs


### Step 3: Retrieve relevant context

`client.search_long_term_memory(...)` searches our stored memories for knowledge similar to the `text` parameter provided. The minimum level of similarity is controlled by the `distance_threshold` parameter.

`client.memory_prompt(...)` performs a similar search, and formats the memories to be ready to be passed to an LLM prompt.

   

In [6]:
# Wait for memories to be indexed (embedding + Redis indexing happens asyncronously)
result = await client.search_long_term_memory(
    text="travel preferences",
    limit=5,
    user_id={"eq": USER_ID},
    namespace={"eq": NAMESPACE},
    distance_threshold=0.65,
)

print("Memories associated with the user's travel preferences are:")
for memory in result.memories:
    print(memory)

Memories associated with the user's travel preferences are:
id='01KN404HPRMXN4WMFJ8A0XD4BX' text='Comfort travel - middle tier (not luxury, not budget)' session_id=None user_id='nitin' namespace='travel_agent' last_accessed=datetime.datetime(2026, 4, 1, 7, 46, 42, 10000, tzinfo=TzInfo(0)) created_at=datetime.datetime(2026, 4, 1, 7, 46, 42, 10000, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2026, 4, 1, 7, 46, 42, 10000, tzinfo=TzInfo(0)) topics=['travel', 'preferences'] entities=[] memory_hash='32702233bbeefd16bee2def9a3b2a8e39f4d82859dcfbf09e6770cd67eac5063' discrete_memory_extracted='t' memory_type=<MemoryTypeEnum.SEMANTIC: 'semantic'> persisted_at=None extracted_from=[] event_date=None dist=0.517617821693 score=0.48238217830700003 score_type=<SearchScoreTypeEnum.SEMANTIC: 'semantic'>
id='01KN404HPRMXN4WMFJ8A0XD4BZ' text='Extra leg room on flights (premium economy or exit row and budget allows, anything goes)' session_id=None user_id='nitin' namespace='travel_agent' last_accessed=d

In [7]:
# Perform a search and format the results for an LLM prompt
user_query = "What are my travel preferences?"

print(f"\nQuery: '{user_query}'")
print(f"Searching memories for user: {USER_ID}\n")

# Note: The memory_prompt() endpoint is designed to return ready-to-use messages for an LLM.
# The memories are pre-formatted into the system prompt so you can directly
# pass messages to OpenAI/Claude without additional processing.
prompt_result = await client.memory_prompt(
    session_id=SESSION_ID,
    query=user_query,
    namespace=NAMESPACE,
    user_id=USER_ID,
    long_term_search={
        "limit": 5,
        # distance_threshold: Lower = stricter when set. If omitted, the server
        # uses no distance filter (distance_threshold=None) for broader KNN recall.
        "user_id": {"eq": USER_ID}  # Only search Nitin's memories
    }
)

messages = prompt_result.get("messages", [])

# Count long-term memories from the structured response
# The memory_prompt() API returns both formatted messages AND a long_term_memories list
long_term_memories = prompt_result.get("long_term_memories") or []
memory_count = len(long_term_memories)

print(f"Long-term memories found: {memory_count}\n")

# Display the messages
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", {})
    if isinstance(content, dict):
        text = content.get("text", str(content))
    else:
        text = str(content)
    print(f"[{role.upper()}]")
    print(f"{text}\n")


Query: 'What are my travel preferences?'
Searching memories for user: nitin

Long-term memories found: 5

[SYSTEM]
## Long term memories related to the user's query
 - User prefers mid-tier, comfortable travel options — neither luxury nor budget. (ID: 01KN4053VZJDR3JVZXTB8EFDS4)
- User prefers hotels with good amenities. (ID: 01KN405EQYH49DTMQR1P3ZZPP2)
- User prefers extra legroom on flights — favors premium economy or an exit-row seat and is willing to pay whatever it takes. (ID: 01KN405B8N63K23PD4GJEQXT8Y)
- User enjoys technology, sports, the outdoors, and innovation hubs. (ID: 01KN405HK849FC7H1TQMYBF5NF)
- User follows a vegetarian diet and needs vegetarian restaurant options/recommendations. (ID: 01KN40574KVK6JKHN94WBBV2PK)

[USER]
What are my travel preferences?



### OpenAI assistant turn (memory + LLM)

The messages from `memory_prompt()` are formatted for chat models. Below we pass them to the OpenAI API, get a reply, and append the user/assistant pair to working memory so later turns stay in context.

### Step 4: Store conversation in working memory

Working memory stores the current conversation for this session.
This is separate from long-term memory (which stores facts/preferences).

In [8]:
messages = [
    MemoryMessage(role="user", content="I'm planning a trip to Japan next month!", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="user", content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Perfect! I'll remember your interest in Mount Fuji. For vegetarian ramen, Kyoto has excellent options.", created_at=datetime.now(timezone.utc))
]

updated_memory = WorkingMemory(
    session_id=SESSION_ID, namespace=NAMESPACE, messages=messages, user_id=USER_ID
)

response = await client.put_working_memory(SESSION_ID, updated_memory)
print(f"Working memory updated with {len(messages)} messages")
print(f"Session: {SESSION_ID}")
print(response)

Working memory updated with 4 messages
Session: nitin-travel-session
messages=[MemoryMessage(role='user', content="I'm planning a trip to Japan next month!", id='01KN4097AE5KBWSH7CJ5ZBADDR', created_at=datetime.datetime(2026, 4, 1, 7, 49, 15, 214623, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content='Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!', id='01KN4097AE5KBWSH7CJ5ZBADDS', created_at=datetime.datetime(2026, 4, 1, 7, 49, 15, 214715, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='user', content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen.", id='01KN4097AE5KBWSH7CJ5ZBADDT', created_at=datetime.datetime(2026, 4, 1, 7, 49, 15, 214733, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content="Perfect! I'll remember your interest in Mount Fu

Let's search and see what's actually stored in Redis

In [10]:
# Search all memories for this user
all_memories = await client.search_long_term_memory(
    text="travel preferences",  # Broad search
    namespace={"eq": "travel_agent"},
    user_id={"eq": "nitin"},
    limit=20,
)

print(f"\nFound {all_memories.total} memories:\n")

for idx, memory in enumerate(all_memories.memories, 1):
    # Calculate relevance score (1 - distance)
    relevance = (1 - memory.dist) * 100 if memory.dist else 0

    print(f"{idx}. [{relevance:.0f}% relevant]")
    print(f"   ID: {memory.id}")
    print(f"   Text: {memory.text}")
    print(f"   Topics: {memory.topics}")
    print(f"   Created: {memory.created_at}\n")


Found 9 memories:

1. [50% relevant]
   ID: 01KN4053VZJDR3JVZXTB8EFDS4
   Text: User prefers mid-tier, comfortable travel options — neither luxury nor budget.
   Topics: ['travel', 'preferences']
   Created: 2026-04-01 07:46:42.010000+00:00

2. [43% relevant]
   ID: 01KN405B8N63K23PD4GJEQXT8Y
   Text: User prefers extra legroom on flights — favors premium economy or an exit-row seat and is willing to pay whatever it takes.
   Topics: ['travel', 'preferences', 'airline']
   Created: 2026-04-01 07:46:42.010000+00:00

3. [37% relevant]
   ID: 01KN405EQYH49DTMQR1P3ZZPP2
   Text: User prefers hotels with good amenities.
   Topics: ['travel', 'preferences', 'accommodation']
   Created: 2026-04-01 07:46:42.010000+00:00

4. [34% relevant]
   ID: 01KN405HK849FC7H1TQMYBF5NF
   Text: User enjoys technology, sports, the outdoors, and innovation hubs.
   Topics: ['sports', 'interests', 'preferences', 'travel', 'technology']
   Created: 2026-04-01 07:46:42.010000+00:00

5. [33% relevant]
   ID: 01K

# Pattern 2: Background Extraction

Now that you've seen how to manually store and retrieve memories let's look at the real strength of the Redis Agent Memory, automatic memory extraction.

This is the fastest way to show AMS learning from conversations without manual memory creation.

Just store conversations - system extracts memories automatically


### Step 1: Create session with extraction strategy configured

In [11]:
from agent_memory_client.models import MemoryStrategyConfig


SESSION_ID_AUTO = "shannon-auto-session"
USER_ID_AUTO = "shannon"

created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    user_id=USER_ID_AUTO,  # Include user_id for consistent session key
    # Configure automatic extraction
    long_term_memory_strategy=MemoryStrategyConfig(
        strategy="discrete"  # Extract individual facts (default)
    ),
)

print(f"Session created with automatic extraction: {SESSION_ID_AUTO}")
strategy = working_memory.long_term_memory_strategy
print(f"Strategy: {strategy.strategy if strategy else 'default'}")

Session created with automatic extraction: shannon-auto-session
Strategy: discrete


### Step 2: Just store the conversation - extraction happens automatically!

In [12]:
conversation = [
    MemoryMessage(role="user", content="I'm Shannon. I'm planning a hiking trip to Japan and need vegetarian food options.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="user", content="I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Noted! I'll remember your preference for comfortable mid-tier accommodations.", created_at=datetime.now(timezone.utc))
]

working_memory_update = WorkingMemory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    messages=conversation,
    user_id=USER_ID_AUTO,
    # Strategy is already configured on the session
)

response = await client.put_working_memory(SESSION_ID_AUTO, working_memory_update)

In [14]:
# pass messages to OpenAI/Claude without additional processing.
prompt_result = await client.memory_prompt(
    session_id=SESSION_ID_AUTO,
    query=user_query,
    namespace=NAMESPACE,
    user_id=USER_ID_AUTO,
    long_term_search={
        "limit": 5,
        # distance_threshold: Lower = stricter when set. If omitted, the server
        # uses no distance filter (distance_threshold=None) for broader KNN recall.
        "user_id": {"eq": USER_ID_AUTO}  # Only search Nitin's memories
    }
)

messages = prompt_result.get("messages", [])

long_term_memories = prompt_result.get("long_term_memories") or []

print(f"Long-term memories found: {memory_count}")

# Display the messages
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", {})
    if isinstance(content, dict):
        text = content.get("text", str(content))
    else:
        text = str(content)
    print(f"[{role.upper()}]")
    print(text)
    print()


Long-term memories found: 5
[USER]
I'm Shannon. I'm planning a hiking trip to Japan and need vegetarian food options.

[ASSISTANT]
Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine.

[USER]
I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget.

[ASSISTANT]
Noted! I'll remember your preference for comfortable mid-tier accommodations.

[SYSTEM]
## Long term memories related to the user's query
 - User's accommodation preferences depend on the budget (ID: 01KN40GMBN4QREEBG9EA4S1D2B)
- User prefers nice hotels with good amenities that are comfortable but not too fancy (ID: 01KN40GMBN4QREEBG9EA4S1D2A)
- User is planning a hiking trip to Japan (ID: 01KN40GMBN4QREEBG9EA4S1D28)
- User needs vegetarian food options (ID: 01KN40GMBN4QREEBG9EA4S1D29)

[USER]
What are my travel preferences?



### Background Extraction Summary

**Where memory operations happen:**
- `get_or_create_working_memory()` configures extraction strategy
- `put_working_memory()` stores the conversation
- the background worker extracts and stores long-term memories

# Putting it all together with an LLM
Now that we've seen how automatic memory extraction works, we can combine it with an LLM to create an agent that creates structured memories naturally from conversation.

Below we use the **same Shannon session** from background extraction (`SESSION_ID_AUTO`): each turn calls `memory_prompt()` to pull working-memory history plus relevant long-term facts, sends that transcript to the OpenAI Chat Completions API, then **appends** the user and assistant utterances with `append_messages_to_working_memory()` so the server can keep learning from the thread.


In [15]:
from openai import AsyncOpenAI

OPENAI_CHAT_MODEL = os.environ.get("OPENAI_CHAT_MODEL", "gpt-4o-mini")

SYSTEM_PROMPT = """You are a helpful assistant with access to a user's remembered preferences.
Use remembered facts when they are relevant, but do not mention internal tools or memory systems.
If the user is sharing preferences or personal facts, acknowledge them briefly and respond naturally.
Do not invent preferences that are not present in the provided memories.
"""

SESSION_ID_TRAVEL = "travel_assistant"
USER_ID_TRAVEL = "justin"

llm_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


def _message_text(message):
    content = message.get("content")
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    return str(content)


async def show_transcript():
    print( "Showing conversation transcript...")
    print('========================================')
    prompt_result = await client.memory_prompt(
        session_id=SESSION_ID_TRAVEL,
        query=" ",
        namespace=NAMESPACE,
        user_id=USER_ID_TRAVEL,
        long_term_search={"user_id": {"eq": USER_ID_TRAVEL}}
    )
    for m in prompt_result.get("messages", []):
        print(f"{m['role']}: {_message_text(m)}")
    print('========================================')


async def show_memories():
    print(f"Showing memories for {USER_ID_TRAVEL}")
    print('========================================')
    result = await client.search_long_term_memory(
        text=" ",  # Broad search
        namespace={"eq": NAMESPACE},
        user_id={"eq": USER_ID_TRAVEL},
        limit=20,
    )
    for memory in result.memories:
        topics = memory.topics if memory.topics else []
        print(f"{memory.text}\n  topics: {topics}\n")
    print('========================================')

In [16]:
from IPython.display import display, Markdown
import time


while True:
    try:
        # Get user input
        user_input = input("You: ")

        # Check for the exit condition
        if user_input.lower() == '/exit':
            print("Chat session ended.")
            break

        if user_input.lower() == '/transcript':
            print(await show_transcript())
            continue

        if user_input.lower() == '/memories':
            print(await show_memories())
            continue

        # get relevant memories from working_memory
        prompt_result = await client.memory_prompt(
            session_id=SESSION_ID_TRAVEL,
            query=user_input,
            namespace=NAMESPACE,
            user_id=USER_ID_TRAVEL,
            long_term_search={
                "limit": 5,
                # distance_threshold: Lower = stricter when set. If omitted, the server
                # uses no distance filter (distance_threshold=None) for broader KNN recall.
                "user_id": {"eq": USER_ID_TRAVEL}  # Only search Shannon's memories
            }
        )

        display(Markdown(user_input))

        response = await llm_client.chat.completions.create(
            model=OPENAI_CHAT_MODEL,
            messages=[
                {"role": m["role"], "content": _message_text(m)} for m in (prompt_result.get("messages") or [])
            ]
            + [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_input}],
            max_completion_tokens=500,
            temperature=0.2,
        )
        response = response.choices[0].message.content.strip()

        display(Markdown(response))

        # append response to working_memory
        await client.append_messages_to_working_memory(messages=[
            MemoryMessage(role="user", content=user_input, created_at=time.time()),
            MemoryMessage(role="assistant", content=response, created_at=time.time()),
            ],
        session_id=SESSION_ID_TRAVEL,
        namespace=NAMESPACE,
        user_id=USER_ID_TRAVEL,
        )

    except KeyboardInterrupt:
        # Allows interrupting the loop via the Kernel menu or stop button
        print("\nChat session interrupted.")
        break
    except Exception as e:
        print(f"An error occurred: {e}")
        break

i'm looking to plan a vacation. ideally somewhere tropical as i like surfing

That sounds like a fantastic idea! There are several tropical destinations known for great surfing. Here are a few options you might consider:

1. **Hawaii** - Particularly the North Shore of Oahu, which is famous for its big waves and surf culture. There are spots for all skill levels.

2. **Bali, Indonesia** - Known for its beautiful beaches and consistent surf conditions. Places like Uluwatu and Canggu are popular among surfers.

3. **Costa Rica** - With both Pacific and Caribbean coasts, you can find great surf spots like Tamarindo and Jaco. Plus, the natural beauty is stunning.

4. **Maldives** - Offers incredible surf breaks and beautiful scenery. It’s a bit more remote, but the experience is often worth it.

5. **Puerto Rico** - Known as the "Surfing Capital of the Caribbean," places like Rincon offer great waves and a vibrant surf community.

Do any of these destinations sound appealing to you?

my roommate says bali is great. tell me more about that

Bali is indeed a fantastic destination, especially for surfers! Here are some highlights about Bali that you might find appealing:

1. **Surf Spots**: Bali has a variety of surf breaks suitable for all skill levels. Some of the most popular spots include:
   - **Uluwatu**: Known for its powerful waves and stunning cliffside views.
   - **Kuta Beach**: Great for beginners, with consistent waves and a lively atmosphere.
   - **Canggu**: Offers a mix of beach breaks and reef breaks, popular among both beginners and advanced surfers.

2. **Surf Schools and Rentals**: There are numerous surf schools and rental shops throughout Bali, making it easy to take lessons or rent gear if you need it.

3. **Stunning Beaches**: Beyond surfing, Bali is home to beautiful beaches, such as Seminyak, Nusa Dua, and Padang Padang, where you can relax and enjoy the sun.

4. **Culture and Scenery**: Bali is rich in culture, with beautiful temples, traditional ceremonies, and lush landscapes. Don’t miss places like Ubud for its art scene and rice terraces.

5. **Nightlife and Dining**: The island has a vibrant nightlife, especially in areas like Seminyak and Kuta, with plenty of restaurants, bars, and beach clubs to explore.

6. **Wellness and Relaxation**: Bali is also known for its wellness retreats, yoga studios, and spas, offering a great way to unwind after a day of surfing.

Overall, Bali offers a perfect blend of adventure, relaxation, and culture, making it a popular choice for travelers. If you decide to go, you’re likely to have an amazing time!

i am still a beginner so i will need surf lessons

That’s great! Bali is an excellent place to take surf lessons, especially for beginners. Here are a few tips on finding surf lessons in Bali:

1. **Surf Schools**: There are many reputable surf schools in Bali that cater to beginners. Look for schools with experienced instructors and good reviews. Some popular ones include:
   - **Odyssey Surf School**: Known for its friendly instructors and personalized lessons.
   - **Bali Learn To Surf**: Offers group and private lessons, focusing on safety and skill development.
   - **Rip Curl School of Surf**: A well-established school with a good reputation.

2. **Group vs. Private Lessons**: Group lessons can be a fun way to meet other surfers and share the experience, while private lessons offer more personalized attention.

3. **Equipment Rental**: Most surf schools provide all the necessary equipment, including boards and wetsuits, so you won’t have to worry about bringing your own.

4. **Best Time to Learn**: The dry season (April to October) typically offers more consistent waves and better weather, making it an ideal time for beginners.

5. **Safety First**: Make sure to choose a school that prioritizes safety and provides proper instruction on surf etiquette and ocean awareness.

With the right lessons and guidance, you’ll be catching waves in no time! Enjoy your surfing adventure in Bali!

Showing conversation transcript...
user: i'm looking to plan a vacation. ideally somewhere tropical as i like surfing
assistant: That sounds like a fantastic idea! There are several tropical destinations known for great surfing. Here are a few options you might consider:

1. **Hawaii** - Particularly the North Shore of Oahu, which is famous for its big waves and surf culture. There are spots for all skill levels.

2. **Bali, Indonesia** - Known for its beautiful beaches and consistent surf conditions. Places like Uluwatu and Canggu are popular among surfers.

3. **Costa Rica** - With both Pacific and Caribbean coasts, you can find great surf spots like Tamarindo and Jaco. Plus, the natural beauty is stunning.

4. **Maldives** - Offers incredible surf breaks and beautiful scenery. It’s a bit more remote, but the experience is often worth it.

5. **Puerto Rico** - Known as the "Surfing Capital of the Caribbean," places like Rincon offer great waves and a vibrant surf community.

Do any o

## Cleanup

Optional: remove demo sessions after the walkthrough.


In [17]:
# Cleanup demo data (optional)
cleanup = True  # Set to True to delete demo data
from agent_memory_client.models import ForgetPolicy

if cleanup:
    sessions_to_delete = await client.list_sessions()
    for sid in sessions_to_delete.sessions:
        try:
            await client.delete_working_memory(sid, namespace="travel_agent")

            await client.forget_long_term_memories(
                policy=ForgetPolicy(budget=0),
                session_id="my-session",
                dry_run=False,
                limit=1000,
            )
            print(f"Deleted session: {sid}")
        except Exception:
            print(f"Error deleting session: {sid}")
            pass
else:
    print("Cleanup skipped. Set cleanup=True to delete demo data.")

Deleted session: nitin-travel-session
Deleted session: shannon-auto-session
Deleted session: travel_assistant
